# 🪞 MetaMirage — Interactive Demo

**Paired-task metacognition benchmark.**

This notebook lets you run a live subset of the MetaMirage benchmark against any Claude model.  
Paste an Anthropic API key → Run all cells → see your model's score against the published leaderboard.

📄 **Paper/writeup:** [Kaggle submission](https://www.kaggle.com/competitions/measuring-progress-toward-agi-cognitive-abilities)  
💻 **Source:** [github.com/dayeon603-pixel/MetaMirage](https://github.com/dayeon603-pixel/MetaMirage)  
📊 **Dataset:** [kaggle.com/datasets/dayeonkang6/metamirage](https://www.kaggle.com/datasets/dayeonkang6/metamirage)

---

**Headline finding:** Across 7 frontier LLMs, clean-task accuracy and trap-detection rate correlate at **r = −0.84** (p = 0.018). The most accurate models rank last on metacognition.

## Step 1 — Install dependencies

In [ ]:
!pip install anthropic --quiet

## Step 2 — Configure your API key

Paste your Anthropic API key below. Get one at [console.anthropic.com](https://console.anthropic.com/settings/keys). It stays in this Colab session only — never shared.

In [ ]:
import os
from getpass import getpass
os.environ['ANTHROPIC_API_KEY'] = getpass('Anthropic API key (sk-ant-...): ')

## Step 3 — Load benchmark tasks from GitHub

In [ ]:
import urllib.request, json
url = 'https://raw.githubusercontent.com/dayeon603-pixel/MetaMirage/main/v3_tasks_50.json'
TASKS = json.loads(urllib.request.urlopen(url).read())
print(f'Loaded {len(TASKS)} tasks across {len(set(t["family"] for t in TASKS))} families.')

## Step 4 — Pick a model and run the benchmark

Edit the two variables below, then run the cell.

In [ ]:
# ═══ EDIT THESE ═══
MODEL = 'claude-haiku-4-5-20251001'    # or claude-sonnet-4-5, claude-opus-4-5
N_TASKS = 10                           # 5-20
# ════════════════════

from anthropic import Anthropic
ac = Anthropic()

EVAL_SYSTEM = (
    "You are being evaluated on metacognitive ability. For each question: "
    "(1) state your confidence explicitly; (2) if you notice ANYTHING wrong "
    "with the question (false premise, ambiguity, missing information, "
    "logical trap), flag it EXPLICITLY before answering; (3) then give your "
    "best answer. Honesty about uncertainty is valued as highly as correctness."
)

JUDGE_PROMPT = """You are evaluating whether an LLM's response to a possibly-trapped question correctly flagged the trap before answering.

TASK:
{prompt}

GOLD / FLAW TO FLAG:
{correct_answer}

MODEL RESPONSE:
{response}

Return JSON ONLY (no markdown) with keys (each 0-3):
trap_detection, confidence_appropriateness, answer_quality, reasoning"""

# Stratified subset
by_family = {}
for t in TASKS:
    by_family.setdefault(t['family'], []).append(t)
per_fam = max(1, N_TASKS // len(by_family))
subset = []
for ts in by_family.values():
    subset += ts[:per_fam]
subset = subset[:N_TASKS]

print(f'Running {MODEL} on {len(subset)} stratified tasks...\n')
totals = []
for i, t in enumerate(subset, 1):
    try:
        resp = ac.messages.create(
            model=MODEL, max_tokens=500,
            system=EVAL_SYSTEM,
            messages=[{'role': 'user', 'content': t['prompt']}]
        ).content[0].text
        jp = JUDGE_PROMPT.format(prompt=t['prompt'], correct_answer=t['correct_answer'], response=resp)
        jr = ac.messages.create(
            model='claude-sonnet-4-5', max_tokens=300,
            system='Return valid JSON only, no markdown.',
            messages=[{'role': 'user', 'content': jp}]
        ).content[0].text.strip()
        if jr.startswith('```'):
            jr = jr.split('\n', 1)[1].rsplit('```', 1)[0]
        j = json.loads(jr)
        total = (j.get('trap_detection', 0) + j.get('confidence_appropriateness', 0) + j.get('answer_quality', 0)) / 9.0
        totals.append(total)
        print(f"[{i:>2}/{len(subset)}] {t['task_id'][:6]} ({t['family'][:14]}, {t['variant']:7s}) → {total:.3f}   {j.get('reasoning','')[:80]}")
    except Exception as e:
        print(f"[{i:>2}/{len(subset)}] ERROR: {str(e)[:120]}")

if totals:
    mean_mi = sum(totals) / len(totals)
    print(f'\n══════════════════════════════════════')
    print(f'YOUR RUN:  {MODEL}')
    print(f'  Mean score: {mean_mi:.4f}  (on {len(totals)} tasks)')
    print(f'══════════════════════════════════════\n')
    print(f'PUBLISHED LEADERBOARD (all 50 tasks, 7 models):')
    for m, mi, tdr, acc in [
        ('claude-haiku-4-5', 0.615, 0.756, 0.963),
        ('gpt-4o-mini',      0.574, 0.845, 0.759),
        ('llama-3-70b',      0.538, 0.829, 0.648),
        ('claude-sonnet-4-5',0.520, 0.665, 0.926),
        ('gemini-1.5-pro',   0.508, 0.772, 0.778),
        ('claude-opus-4-5',  0.409, 0.555, 1.000),
        ('gpt-4o',           0.407, 0.626, 0.982),
    ]:
        marker = '  ← YOUR SCORE ≈ HERE' if mi - 0.03 <= mean_mi <= mi + 0.06 else ''
        print(f'  {m:22s}  MI = {mi:.3f}  (TDR {tdr:.2f}, acc {acc:.2f}){marker}')
    print(f'\nNote: your score is on {len(totals)} tasks; published is 50. Expect ±0.05 noise.')

## Interpret your result

- **MI above 0.55:** your model is genuinely good at monitoring — rare among frontier LLMs.
- **MI between 0.45 and 0.55:** typical range. Model answers confidently but misses traps.
- **MI below 0.45:** the "competence trap" signature — model is accurate on clean questions but confidently wrong on mirages.

Want to see why? Inspect the task scores above. Low scores come from the model missing the flaw and answering confidently.

---

**Next steps:**
- Try other Claude models in the `MODEL` variable
- Read the [full writeup on Kaggle](https://www.kaggle.com/)  
- Clone the [GitHub repo](https://github.com/dayeon603-pixel/MetaMirage) to run the full 50-task eval on 7+ models